# Stress Score Prediction V7 — Dual Quantile Forest Ensemble

V6 모델과 나이·건강지표 상호작용 모델을 50:50으로 결합한 제출용 앙상블입니다.
모든 전처리와 피처 생성은 다른 행이나 Test 통계를 사용하지 않습니다.


## 0. Import & Data Load


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold


In [2]:
DATA_DIR = Path('/Users/bitsaem/Desktop/stress_project/open')

train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')

print('Train shape:', train.shape)
print('Test shape:', test.shape)


Train shape: (3000, 18)
Test shape: (3000, 17)


## 1. Data Split


In [3]:
TARGET = 'stress_score'
ID_COL = 'ID'

X = train.drop(columns=[TARGET, ID_COL])
y = train[TARGET].copy()

X_test = test.drop(columns=[ID_COL])
test_ids = test[ID_COL].copy()


## 2. Leakage-safe Feature Engineering


In [4]:
def add_base_features(df):
    """각 행 내부의 값만 사용해 건강 관련 파생변수를 생성합니다."""
    df = df.copy()

    height_m = df['height'] / 100
    sbp = df['systolic_blood_pressure']
    dbp = df['diastolic_blood_pressure']

    df['bmi'] = df['weight'] / (height_m ** 2)
    df['pulse_pressure'] = sbp - dbp
    df['mean_arterial_pressure'] = (sbp + 2 * dbp) / 3
    df['blood_pressure_ratio'] = sbp / (dbp + 1e-6)
    df['cholesterol_glucose_ratio'] = (
        df['cholesterol'] / (df['glucose'] + 1e-6)
    )
    df['missing_count'] = df.isna().sum(axis=1)

    return df


def add_weighted_copies(df, features):
    """중요 피처의 ExtraTrees 내 무작위 선택 확률을 높입니다."""
    df = df.copy()
    for feature in features:
        df[f'{feature}__copy1'] = df[feature]
    return df


V6_WEIGHTED_FEATURES = [
    'mean_working',
    'bmi',
    'cholesterol',
    'height',
    'glucose',
    'weight',
    'cholesterol_glucose_ratio',
    'bone_density',
]

AGE_WEIGHTED_FEATURES = V6_WEIGHTED_FEATURES + [
    'mean_arterial_pressure',
    'diastolic_blood_pressure',
]


In [5]:
def make_v6_features(df):
    df = add_base_features(df)
    return add_weighted_copies(df, V6_WEIGHTED_FEATURES)


def make_age_features(df):
    df = add_base_features(df)

    df['age_bmi'] = df['age'] * df['bmi']
    df['age_glucose'] = df['age'] * df['glucose']
    df['age_cholesterol'] = df['age'] * df['cholesterol']
    df['age_bone_density'] = df['age'] * df['bone_density']
    df['age_mean_arterial_pressure'] = (
        df['age'] * df['mean_arterial_pressure']
    )

    return add_weighted_copies(df, AGE_WEIGHTED_FEATURES)


X_v6 = make_v6_features(X)
X_test_v6 = make_v6_features(X_test)

X_age = make_age_features(X)
X_test_age = make_age_features(X_test)

print('V6 feature shape:', X_v6.shape)
print('Age feature shape:', X_age.shape)


V6 feature shape: (3000, 30)
Age feature shape: (3000, 37)


## 3. Leakage-safe Preprocessing & Model Factory


In [6]:
def make_quantile_forest_pipeline(X_data):
    categorical_cols = X_data.select_dtypes(
        include=['object', 'category']
    ).columns.tolist()

    numerical_cols = X_data.select_dtypes(
        include=['number']
    ).columns.tolist()

    categorical_pipeline = Pipeline([
        (
            'imputer',
            SimpleImputer(
                strategy='constant',
                fill_value='missing'
            )
        ),
        (
            'encoder',
            OneHotEncoder(
                handle_unknown='ignore',
                sparse_output=False
            )
        )
    ])

    numerical_pipeline = Pipeline([
        (
            'imputer',
            SimpleImputer(
                strategy='median',
                add_indicator=True
            )
        )
    ])

    preprocessor = ColumnTransformer([
        ('categorical', categorical_pipeline, categorical_cols),
        ('numerical', numerical_pipeline, numerical_cols)
    ])

    model = ExtraTreesRegressor(
        n_estimators=1200,
        min_samples_leaf=1,
        max_features=1,
        random_state=42,
        n_jobs=-1
    )

    return Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])


pipeline_v6 = make_quantile_forest_pipeline(X_v6)
pipeline_age = make_quantile_forest_pipeline(X_age)


In [7]:
def quantile_forest_predict(
    fitted_pipeline,
    X_data,
    quantile
):
    transformed = fitted_pipeline.named_steps[
        'preprocessor'
    ].transform(X_data)

    forest = fitted_pipeline.named_steps['model']

    tree_predictions = np.column_stack([
        tree.predict(transformed)
        for tree in forest.estimators_
    ])

    prediction = np.quantile(
        tree_predictions,
        quantile,
        axis=1
    )

    return np.round(prediction, 2)


## 4. Five-fold Cross-validation


In [8]:
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

v6_oof = np.zeros(len(X))
age_oof = np.zeros(len(X))

for fold, (train_idx, valid_idx) in enumerate(cv.split(X), start=1):
    fold_v6 = clone(pipeline_v6)
    fold_v6.fit(
        X_v6.iloc[train_idx],
        y.iloc[train_idx]
    )
    v6_oof[valid_idx] = quantile_forest_predict(
        fold_v6,
        X_v6.iloc[valid_idx],
        quantile=0.5125
    )

    fold_age = clone(pipeline_age)
    fold_age.fit(
        X_age.iloc[train_idx],
        y.iloc[train_idx]
    )
    age_oof[valid_idx] = quantile_forest_predict(
        fold_age,
        X_age.iloc[valid_idx],
        quantile=0.5
    )

    print(f'Fold {fold} complete')

v7_oof = np.round(
    0.5 * v6_oof
    + 0.5 * age_oof,
    2
)

v6_mae = np.mean(np.abs(y.to_numpy() - v6_oof))
age_mae = np.mean(np.abs(y.to_numpy() - age_oof))
v7_mae = np.mean(np.abs(y.to_numpy() - v7_oof))

print('V6 CV MAE:', round(v6_mae, 6))
print('Age model CV MAE:', round(age_mae, 6))
print('V7 Ensemble CV MAE:', round(v7_mae, 6))


Fold 1 complete
Fold 2 complete
Fold 3 complete
Fold 4 complete
Fold 5 complete
V6 CV MAE: 0.149243
Age model CV MAE: 0.148847
V7 Ensemble CV MAE: 0.148857


## 5. Final Training & Prediction


In [9]:
pipeline_v6.fit(X_v6, y)
pipeline_age.fit(X_age, y)

pred_v6 = quantile_forest_predict(
    pipeline_v6,
    X_test_v6,
    quantile=0.5125
)

pred_age = quantile_forest_predict(
    pipeline_age,
    X_test_age,
    quantile=0.5
)

pred_v7 = np.round(
    0.5 * pred_v6
    + 0.5 * pred_age,
    2
)
pred_v7 = np.clip(pred_v7, 0, 1)

print('Prediction shape:', pred_v7.shape)
print('Prediction range:', pred_v7.min(), pred_v7.max())


Prediction shape: (3000,)
Prediction range: 0.0 1.0


## 6. Submission


In [10]:
submission_v7 = submission.copy()
submission_v7[ID_COL] = test_ids
submission_v7[TARGET] = pred_v7

output_path_v7 = (
    DATA_DIR.parent
    / 'submit_v7_dual_quantile_forest.csv'
)

submission_v7.to_csv(
    output_path_v7,
    index=False
)

print('Saved:', output_path_v7)
submission_v7.head()


Saved: /Users/bitsaem/Desktop/stress_project/submit_v7_dual_quantile_forest.csv


,ID,stress_score
0,TEST_0000,0.57
1,TEST_0001,0.97
2,TEST_0002,0.19
3,TEST_0003,0.44
4,TEST_0004,0.53
